# Ejercicio 7 — indicadores y visualizaciones con DuckDB

Este notebook sustituye el tablero de Metabase. Prepara las vistas persistentes, ejecuta las consultas SQL versionadas y presenta los resultados y gráficas dentro de Jupyter. Requiere que `data/processed/ejercicio_6.duckdb` ya haya sido creado por el ejercicio 6.

## 1. Configuración

In [3]:
from pathlib import Path
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATABASE = ROOT / 'data' / 'processed' / 'ejercicio_6.duckdb'
assert DATABASE.is_file(), (
    f'No existe {DATABASE}. Ejecute primero el ejercicio 6 o el script de materialización.'
)
print(f'Proyecto: {ROOT}')
print(f'Base: {DATABASE}')

AssertionError: No existe c:\Users\aripi\Documents\duckdb-main-ejercicios-7-8-jupyter\duckdb-main\data\processed\ejercicio_6.duckdb. Ejecute primero el ejercicio 6 o el script de materialización.

## 2. Preparación de las vistas

Esta celda requiere acceso de escritura momentáneo. Si Metabase está abierto con escritura, deténgalo antes de ejecutarla.

In [ ]:
sql_vistas = (ROOT / 'ejercicio_7_vistas.sql').read_text(encoding='utf-8')
with duckdb.connect(str(DATABASE)) as con_escritura:
    con_escritura.execute('BEGIN TRANSACTION')
    try:
        con_escritura.execute(sql_vistas)
        con_escritura.execute('COMMIT')
    except Exception:
        con_escritura.execute('ROLLBACK')
        raise
print('Vistas viajes_base y viajes_analiticos preparadas.')

In [ ]:
con = duckdb.connect(str(DATABASE), read_only=True)

def consultar(nombre):
    sql = (ROOT / nombre).read_text(encoding='utf-8')
    return con.execute(sql).df()

## 3. Indicadores principales

In [ ]:
archivos_kpi = {
    'Viajes válidos': 'ejercicio_7_kpi_viajes.sql',
    'Ingresos': 'ejercicio_7_kpi_ingresos.sql',
    'Ticket medio': 'ejercicio_7_kpi_ticket_medio.sql',
    'Ticket mediano': 'ejercicio_7_kpi_ticket_mediano.sql',
    'Distancia mediana': 'ejercicio_7_kpi_distancia_mediana.sql',
    'Duración media': 'ejercicio_7_kpi_duracion_media.sql',
}
kpis = pd.DataFrame([
    {'indicador': etiqueta, 'valor': consultar(archivo).iloc[0, 0]}
    for etiqueta, archivo in archivos_kpi.items()
])
display(kpis.style.format({'valor': '{:,.2f}'}))

## 4. Evolución mensual

In [ ]:
mensuales = consultar('ejercicio_7_viajes_mensuales.sql')
mensuales['periodo'] = pd.to_datetime(mensuales['periodo'])
display(mensuales.head())

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for taxi, datos in mensuales.groupby('tipo_taxi'):
    axes[0].plot(datos['periodo'], datos['viajes'], marker='o', label=taxi)
    axes[1].plot(datos['periodo'], datos['ingresos'], marker='o', label=taxi)
axes[0].set_title('Viajes mensuales')
axes[0].set_ylabel('Viajes')
axes[1].set_title('Ingresos mensuales')
axes[1].set_ylabel('Importe total')
for ax in axes:
    ax.set_xlabel('Periodo')
    ax.grid(alpha=.25)
    ax.legend(title='Taxi')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 5. Demanda por día y hora

In [ ]:
demanda = consultar('ejercicio_7_demanda_horaria.sql')
orden = ['Lunes', 'Martes', 'Miercoles', 'Jueves', 'Viernes', 'Sabado', 'Domingo']
matriz = demanda.pivot(index='dia_semana', columns='hora', values='viajes').reindex(orden).fillna(0)
fig, ax = plt.subplots(figsize=(14, 5))
imagen = ax.imshow(matriz.values, aspect='auto', cmap='YlOrRd')
ax.set_yticks(range(len(matriz.index)), matriz.index)
ax.set_xticks(range(len(matriz.columns)), matriz.columns)
ax.set_xlabel('Hora')
ax.set_ylabel('Día')
ax.set_title('Demanda horaria de viajes válidos')
fig.colorbar(imagen, ax=ax, label='Viajes')
plt.tight_layout()
plt.show()

## 6. Comparaciones, pagos, propinas, zonas y calidad

In [ ]:
resultados = {
    'Comparación de taxis': 'ejercicio_7_comparacion_taxis.sql',
    'Formas de pago': 'ejercicio_7_formas_pago.sql',
    'Propinas': 'ejercicio_7_propinas.sql',
    'Zonas de origen': 'ejercicio_7_zonas_origen.sql',
    'Calidad': 'ejercicio_7_calidad.sql',
}
for titulo, archivo in resultados.items():
    print(f'\n{titulo}')
    display(consultar(archivo))

## 7. Cierre de la conexión

In [ ]:
con.close()
print('Ejercicio 7 ejecutado correctamente.')